# Searching for Correlates in Neural Recordings — Circular Analysis ("Double Dipping")

## Motivation

The purpose of this exercise is to explore how a data-analysis *workflow* can influence
the validity of a study, and to show how simulations can be used to validate an analysis
workflow **before** applying it to real data — or even before collecting data.

We use an example from systems neuroscience. In many studies, researchers record from large
populations of individual cells (neurons) while an animal performs a task, and search for
cells whose activity is correlated with the animal's **behavioral performance** (for
example, its accuracy on a perceptual *discrimination* task). A common initial step may be to group those cells into "cell types" either based on their response patterns or based on measurements in another modality (ie cell morphology from EM), before running additional analysis on such groups.

Here's what might seem like a reasonable workflow following this pattern:

1. Identify the cells whose activity correlates with behavioral discrimination performance.
2. Average the activity across those identified cells (a "group of responsive cells").
3. Test how well this aggregate "responsive cell-group activity" score correlates with
   discrimination performance.

We simulate this experiment and analysis in two cases:

- **ground truth is *no effect*** (cell activity is unrelated to discrimination performance).
- **ground truth is a *real effect*** (discrimination performance is correlated with cell activity).

The key lesson: selecting cells by their correlation with performance and then re-testing
that same correlation on the selected cells produces a spurious "significant" result even
when there is no real effect.

## Roadmap of this notebook

1. **Part 1: spurious results from random data** Walkthrough the outlined analysis workflow on simulated random data, see how convincing the spurious effects can be, then explore how to detect the issue and why it occurred (circular analysis / double-dipping).
2. **Part 2: solutions**: Explore ways of avoiding this pitfall: leave-one-out
   cross-validation (LOO-CV), a 50/50 train/test split, and FDR correction — comparing their
   false-positive rates.
2. **Part 3: simulated real effects** Repeat with a genuine effect and assess the balance between detected true effects, undetected true effects, and spurious detections.
3. **Extension 1: Power calculations.** Sweep effect size and sample size to see how much data each method
   needs.
4. **Extension 2: Permutation test of LOO-CV.** A final refinement that calibrates the cross-validated
   statistic with a label-permutation null.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Set a seed so the notebook is reproducible (remove for fresh random draws)
rng = np.random.default_rng(0)

## Notes on key functions used:

We rely entirely on standard `scipy.stats` functions:

- `scipy.stats.pearsonr` for the per-cell correlation with performance (one cell at a time in
  the single-run analyses, or with `axis=0` in the heavier repeated simulations),
- `scipy.stats.linregress` for the best-fit line in the scatter plots



# Preparations

Suppose we recorded from a **single animal** over one session and split that session into
`Ngroups = 20` groups of trials.

## Simulating behavior

For each trial group we measure the animal's behavioral performance on the discrimination
task, giving a performance score $P$ (e.g. percent correct). To simulate this, we draw a
score for each trial group at random from a normal distribution with mean $\mu_P = 75$ and
standard deviation $\sigma_P = 8$.

(Rounding $P$ to an integer percent is not required — it just mimics a realistic performance
measure.)




In [ ]:
Ngroups = 20
mP, sP = 75, 8
Perf = sP * rng.standard_normal(Ngroups) + mP
Perf = np.round(Perf)  # not required, but make performance an integer percent
Perf


## Simulating the calcium imaging recording

Within the single animal we image `Ncells = 200` cells (neurons). For each trial group we
measure each cell's differential activity (activity in the discrimination condition relative
to a control condition). We keep the activity in a matrix of shape `(Ngroups, Ncells)`.

Each cell also has a **random spatial position** in the imaging field, drawn uniformly on the
unit square $[0, 1]^2$. These positions let us visualize *where* the cells that appear
correlated with performance sit in the tissue.

In the
null case (Question 1), **no** cell's activity depends on performance: every cell is pure
noise drawn from a normal distribution with mean 0 and standard deviation $\sigma_A = 1$.

**Here's the key point:** in Question 1 the cell activity is *completely unrelated to
behavioral discrimination performance in any cell*. Let's see how often we would be fooled
into thinking cell activity and discrimination performance are related, when we know they are
not.




# Question 1 — Standard analysis method, **no real effect**

We replicate a common data-analysis workflow to identify cells whose activity correlates
with behavioral discrimination performance:

1. **Correlate each cell with performance.** For each of the `Ncells` cells, take the
   20 trial groups' differential-activity values and the 20 trial groups' performance scores
   $P$ and test whether they are correlated. This gives `Ncells` $r$ values and $p$ values.
2. **Define the "selected" cells** as all cells *positively* correlated with $P$ under
   the criteria $r > 0.1$ **and** $p < 0.05$.
3. **Average within each trial group** the differential activity of the selected cells to get a
   single "Relevant Cell-Group Activity" (RCGA) value per trial group.
4. **Scatter plot** RCGA versus performance $P$ (1 symbol per trial group, 20 trial groups).
5. **Fit a line** and label the plot with the $R^2$ and $p$ values (done in the plotting cell below).

First, simulate the null data set: every cell's differential activity is pure noise with
mean 0 and std `sA`. Crucially, the cell activity does **not** depend on `Perf`, so there is
no true relationship between cell activity and performance.




In [ ]:

Ncells = 200
# Random 2D spatial position for each cell (uniform on the unit square).
CellXY = rng.uniform(0.0, 1.0, size=(Ncells, 2))

sA = 1  # noise std of each cell's differential activity
DA = sA * rng.standard_normal((Ngroups, Ncells))
print(DA)


In [ ]:
# 1. Correlate each cell's differential activity with discrimination performance,
#    one cell at a time using scipy.stats.pearsonr.
r = np.empty(Ncells)
p = np.empty(Ncells)
for j in range(Ncells):
    r[j], p[j] = stats.pearsonr(DA[:, j], Perf)

# 2. "Selected" cells: positive correlation r > 0.1 AND significant p < 0.05.
SelectedCells = np.where((r > 0.1) & (p < 0.05))[0]

# 3. Average the differential activity of the selected cells within each trial group.
RCGA = DA[:, SelectedCells].mean(axis=1)
print(f"Number of 'selected' cells: {SelectedCells.size}")
print(f"RCGA shape (should be {Ngroups}): {RCGA.shape}")


## Plot the null result

For each trial group, plot Relevant Cell-Group Activity (RCGA) vs. the behavioral
discrimination performance, with the best-fit line and the $R^2$ / p-value of the (circular)
correlation. Even though the ground truth is *no effect*, the scatter typically looks
convincingly significant — that is the double-dipping artifact.


In [ ]:
def annotate_corr(ax, rcga, Perf, title):
    ax.plot(rcga, Perf, "o", color="tab:blue", markerfacecolor="tab:blue")
    fit = stats.linregress(rcga, Perf)
    xl = np.array(ax.get_xlim())
    ax.plot(xl, fit.slope * xl + fit.intercept, "-", color="tab:orange")
    ax.set_xlim(xl)
    ax.set_xlabel("Relevant Cell-Group Activity")
    ax.set_ylabel("Discrimination Performance")
    ax.set_title(title)
    Rsquared = fit.rvalue ** 2
    xlims, ylims = ax.get_xlim(), ax.get_ylim()
    xpos = xlims[0] + 0.05 * (xlims[1] - xlims[0])
    ypos = ylims[1] - 0.08 * (ylims[1] - ylims[0])
    ax.text(xpos, ypos, f"$R^2$={Rsquared:.4f}  P={fit.pvalue:.2e}")


fig, ax = plt.subplots(figsize=(6, 5))
annotate_corr(ax, RCGA, Perf, "Standard Analysis Method, No Real Effect")
plt.tight_layout()
plt.show()


## How often are we fooled? Distribution of $p$ and $R^2$ under the null

A single run either clears the $p < 0.05$ bar or it doesn't — it can't tell us the
*false-positive rate*. To see how badly the circular workflow misbehaves, we repeat the
**null simulation** many times (fresh cell activity each run, the same fixed `Perf`) and
collect the final circular $p$-value and $R^2$ from every run.

If the workflow were valid, the $p$-values would be roughly uniform on $[0, 1]$ and only ~5%
would fall below 0.05. Instead we see the distribution piled up near zero: a large fraction of
purely-null runs look "significant."


In [ ]:
# Repeat the null (no-effect) circular analysis many times and collect p and R^2.
N_null = 2000
pvals_null = []
r2_null = []
for _ in range(N_null):
    DA_n = sA * rng.standard_normal((Ngroups, Ncells))
    res = stats.pearsonr(DA_n, Perf[:, np.newaxis], axis=0)
    sel = np.where((res.statistic > 0.1) & (res.pvalue < 0.05))[0]
    if sel.size == 0:
        continue
    rr, pp = stats.pearsonr(DA_n[:, sel].mean(axis=1), Perf)
    pvals_null.append(pp)
    r2_null.append(rr ** 2)

pvals_null = np.array(pvals_null)
r2_null = np.array(r2_null)
frac_sig = np.mean(pvals_null < 0.05)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(pvals_null, bins=np.linspace(0, 1, 21), color="tab:blue", edgecolor="k")
axes[0].axvline(0.05, color="k", linestyle="--", linewidth=1.2, label="p = 0.05")
axes[0].set_xlabel("Circular p-value")
axes[0].set_ylabel("Count (null runs)")
axes[0].set_title(f"Distribution of p-values under the null\n"
                  f"{frac_sig:.0%} of runs have p < 0.05 (nominal 5%)")
axes[0].legend()

axes[1].hist(r2_null, bins=20, color="tab:blue", edgecolor="k")
axes[1].set_xlabel("Circular $R^2$")
axes[1].set_ylabel("Count (null runs)")
axes[1].set_title("Distribution of $R^2$ under the null")

plt.tight_layout()
plt.show()

print(f"Null circular analysis over {N_null} runs:")
print(f"  fraction with p < 0.05 : {frac_sig:.1%}  (nominal 5%)")
print(f"  median R^2             : {np.median(r2_null):.3f}")


## What to think about (interpreting the null result)

The ground truth here is that cell activity is completely unrelated to discrimination
performance — yet the scatter typically shows a convincing-looking positive trend with a high
$R^2$ and a small $p$ value, and the histograms above confirm this happens on a large fraction
of null runs. Every one of these "significant" results is a **false positive**.

## Question 3 — What is wrong with this workflow?

The flaw is **circular analysis**, also called **"double dipping"** or non-independence
error. The *same* 20 trial groups' data are used twice:

1. First to **select** the cells — we keep exactly those cells that happen to correlate
   positively with $P$ in this particular sample.
2. Then to **test** the correlation — we re-measure the correlation between $P$ and the average
   of those hand-picked cells.

Because the selection step cherry-picks cells whose *noise* happens to align with $P$, the
averaged cell group is guaranteed to correlate with $P$ in the same sample even when no true
relationship exists. The final correlation test is therefore not valid: its null distribution
is not the standard one, so the reported $p$ value is meaningless.

**Better (independent) workflows** — make sure selection and testing do not reuse the same
data in a way that manufactures the correlation. In the sections below we develop three:

- **Cross-validation / data splitting:** select the "responsive" cells using one subset of
  trial groups, then compute RCGA and test its correlation with $P$ on a *held-out* subset
  that played no part in selection.
- **Multiple-comparison correction (FDR):** rather than averaging and re-testing, ask how many
  *individual* cells are significantly correlated with $P$ after controlling the false
  discovery rate across all `Ncells` tests.

We first apply each of these to the null data (this section) to check that they bring the
false-positive rate back to the nominal level; later we confirm on real-effect data
(Question 2) that they still detect a genuine effect.


---

# SOLUTION 1: Leave-one-out cross-validation
Break the circularity by using different sets of trial groups to select the cells and test the correlation.

With only 20 trial groups, **leave-one-out cross-validation (LOO-CV)** is a natural choice:

* **Fold $i$** (for $i = 1 \ldots N_\text{groups}$):
  1. Use the $N_\text{groups} - 1$ *training* trial groups to select cells
     (same criteria as before: $r > 0.1$, $p < 0.05$).
  2. Compute the RCGA for the *held-out* trial group $i$ by averaging the
     activity in those training-derived cells.

After all folds, the held-out RCGA values are assembled into a vector of length
$N_\text{groups}$ and correlated with $P$.  Because no trial group's data influenced both the
cell-selection step and its own RCGA value, this correlation is (mostly) independent.

The helper below implements the loop; it returns `RCGA_cv` (one entry per trial group) and
the number of cells selected in each fold (useful for diagnostics).


In [ ]:
def loo_cv_rcga(DA, Perf):
    """Leave-one-out cross-validated RCGA.

    For each held-out trial group i, cells are selected on the other N-1 trial groups
    using the same criteria as the standard analysis (r > 0.1, p < 0.05),
    then RCGA is computed for trial group i on those held-out cells.

    Returns
    -------
    RCGA_cv : ndarray, shape (Ngroups,)
        Cross-validated RCGA; NaN where no cells were selected in that fold.
    n_cells : list of int
        Number of cells selected in each fold (diagnostic).
    """
    Ngroups = DA.shape[0]
    RCGA_cv = np.full(Ngroups, np.nan)
    n_cells = []

    for i in range(Ngroups):
        train_idx = np.delete(np.arange(Ngroups), i)
        # Correlate every cell with performance on the training folds (vectorized pearsonr).
        res = stats.pearsonr(DA[train_idx], Perf[train_idx][:, np.newaxis], axis=0)
        sel_cells = np.where((res.statistic > 0.1) & (res.pvalue < 0.05))[0]
        n_cells.append(sel_cells.size)
        if sel_cells.size > 0:
            RCGA_cv[i] = DA[i, sel_cells].mean()

    return RCGA_cv, n_cells


In [ ]:
# Apply LOO-CV to the null dataset
RCGA_cv_null, ncells_null = loo_cv_rcga(DA, Perf)

print("Null data (no real effect):")
print(f"  Avg cells selected per fold : {np.mean(ncells_null):.1f}  "
      f"(range {min(ncells_null)}–{max(ncells_null)})")
print(f"  Folds with ≥1 cell selected : {np.sum(np.array(ncells_null) > 0)}/{Ngroups}")


### Comparison plot: circular analysis vs. LOO-CV (null data)

Two panels show the scatter of RCGA vs. discrimination performance $P$ on the **null** data.
On the left is the circular analysis (same trial groups used to select and to test); on the
right is LOO-CV (each trial group's RCGA comes from cells selected on the *other* trial
groups). In the LOO-CV plot only trial groups whose fold produced at least one selected cell
are shown.

**Expected outcome:** the circular panel shows a spurious significant correlation while the
LOO-CV panel shows a weak, non-significant one ($p \gg 0.05$).


In [ ]:
def annotate_corr_cv(ax, rcga, Perf, title):
    """Like annotate_corr but handles NaN entries from LOO-CV."""
    mask = ~np.isnan(rcga)
    rcga_valid, Perf_valid = rcga[mask], Perf[mask]
    ax.plot(rcga_valid, Perf_valid, "o", color="tab:green", markerfacecolor="tab:green")
    fit = stats.linregress(rcga_valid, Perf_valid)
    xl = np.array(ax.get_xlim())
    ax.plot(xl, fit.slope * xl + fit.intercept, "-", color="tab:orange")
    ax.set_xlim(xl)
    ax.set_xlabel("Relevant Cell-Group Activity (CV)")
    ax.set_ylabel("Discrimination Performance")
    ax.set_title(title)
    if Perf_valid.size >= 3:
        Rsq = fit.rvalue ** 2
        xlims, ylims = ax.get_xlim(), ax.get_ylim()
        xpos = xlims[0] + 0.05 * (xlims[1] - xlims[0])
        ypos = ylims[1] - 0.08 * (ylims[1] - ylims[0])
        ax.text(xpos, ypos, f"$R^2$={Rsq:.4f}  P={fit.pvalue:.2e}")


fig, axes = plt.subplots(1, 2, figsize=(13, 5))
annotate_corr(axes[0], RCGA, Perf,
              "Circular — No Real Effect\n(same data used for selection & test)")
annotate_corr_cv(axes[1], RCGA_cv_null, Perf,
                 "LOO-CV — No Real Effect\n(independent selection & test)")
plt.tight_layout()
plt.show()


### A problem with LOO-CV, and a simpler fix: a 50/50 train/test split

LOO-CV removes the *worst* of the circularity, but it has a subtle weakness: the 20 folds
each leave out a single trial group, so any two folds share 18 of their 19 training groups.
The held-out RCGA values are therefore **not independent** of one another, and they were all
selected against the same fixed `Perf`. Feeding the assembled `RCGA_cv` vector into
`stats.pearsonr` — which assumes 20 independent observations — gives an **anti-conservative**
p-value. (We return to fixing this properly with a permutation test at the very end.)

A conceptually simpler and fully independent alternative is a **single 50/50 train/test
split**:

* **Train half** (10 trial groups): select the responsive cells (same criteria, $r > 0.1$,
  $p < 0.05$).
* **Test half** (the other 10 trial groups, untouched during selection): compute RCGA on the
  selected cells and correlate it with $P$.

Because the test half played no part in selection, its correlation test is valid and the
`pearsonr` p-value is trustworthy. The cost is reduced power: selection and testing each use
only half the trial groups.


In [ ]:
def train_test_rcga(DA, Perf, train_frac=0.5, generator=None):
    """Single train/test split: select cells on the training trial groups, then compute and
    return RCGA on the held-out test trial groups (which played no part in selection).

    Returns (rcga_test, Perf_test, selected_cells), or None if no cells were selected.
    """
    Ng = DA.shape[0]
    idx = np.arange(Ng)
    if generator is not None:
        idx = generator.permutation(idx)  # randomize which groups are train vs test
    n_train = int(round(train_frac * Ng))
    train_idx, test_idx = idx[:n_train], idx[n_train:]

    res = stats.pearsonr(DA[train_idx], Perf[train_idx][:, np.newaxis], axis=0)
    sel = np.where((res.statistic > 0.1) & (res.pvalue < 0.05))[0]
    if sel.size == 0:
        return None
    rcga_test = DA[test_idx][:, sel].mean(axis=1)
    return rcga_test, Perf[test_idx], sel


# Apply the 50/50 split to the null data (contiguous first-half / second-half split).
split_null = train_test_rcga(DA, Perf)
if split_null is None:
    print("Null data: no cells selected on the training half.")
else:
    rcga_test_null, Perf_test_null, sel_null = split_null
    r_tt, p_tt = stats.pearsonr(rcga_test_null, Perf_test_null)
    print("Null data — 50/50 train/test split:")
    print(f"  cells selected on training half : {sel_null.size}")
    print(f"  test-half correlation           : r = {r_tt:+.3f}, p = {p_tt:.3f}")

    fig, ax = plt.subplots(figsize=(6, 5))
    annotate_corr(ax, rcga_test_null, Perf_test_null,
                  "Train/Test Split — No Real Effect\n(test half: independent of selection)")
    plt.tight_layout()
    plt.show()


---

## Alternative: Multiple-comparison correction (Benjamini-Hochberg FDR)

The LOO-CV approach breaks the circularity by separating selection and testing. A different
valid strategy avoids the circular *average-and-re-test* step altogether: rather than pooling
selected cells and re-testing their average, we simply ask **how many individual cells are
significantly correlated with performance**, while controlling the **false discovery rate
(FDR)** across all `Ncells` per-cell tests with the **Benjamini-Hochberg** procedure.

- **Null data:** BH at $q = 0.05$ should reject essentially no cells (FDR controlled).
- **Real-effect data:** BH should recover some of the truly performance-correlated cells,
  though with limited power given only 20 trial groups and a weak effect.

The number of BH-significant cells is itself the read-out: if it is $\ge 1$ we would declare a
detected effect. Because each per-cell test uses the full data only *once* (no re-testing of a
selected average), this is not circular.


In [ ]:
# can also implement bonferroni directly by just dividing
def fdr_bh_cells(DA, Perf, q=0.05):
    """Select cells whose correlation with Perf survives BH-FDR at level q.

    Returns the indices of the surviving cells (the FDR-significant discoveries).
    """
    res = stats.pearsonr(DA, Perf[:, np.newaxis], axis=0)
    adj_p = stats.false_discovery_control(res.pvalue, method="bh")
    return np.where(adj_p <= q)[0]


q_fdr = 0.05
fdr_null = fdr_bh_cells(DA, Perf, q=q_fdr)
print(f"FDR-BH (q = {q_fdr}) cells discovered on null data: {fdr_null.size}")


### False-positive rate — all four methods

A single simulation run is not enough to measure the false-positive rate — one run
either passes or fails the significance threshold by chance.  To estimate the actual
false-positive rate we repeat the **null simulation** (no real effect) many times and
count how often each method incorrectly declares a significant result.

The loop below runs `N_iter = 1000` fresh null datasets.  For each:

1. Draw new `DA` (only `Perf` stays fixed — the design is fixed, the cell data are resampled).
2. Run the **circular** analysis and record whether the final $p < \alpha$.
3. Run the **LOO-CV** analysis and record whether the final $p < \alpha$.
4. Run the **train/test split** analysis and record whether the test-half $p < \alpha$.
5. Run the **FDR-BH** analysis and record whether it declares $\ge 1$ significant cell at $q = \alpha$.

At $\alpha = 0.05$ the nominal false-positive rate is 5%.

| Method | Selection / test | Trial groups used for selection | Trial groups used for final test |
|---|---|---|---|
| Circular | Correlation with $P$, average, re-test | All 20 | Same 20 |
| LOO-CV | Correlation with $P$ | 19 (held one out) | All 20 (assembled) |
| Train/test split | Correlation with $P$ | 10 (train half) | 10 (test half) |
| FDR-BH | Per-cell correlation with $P$, BH-corrected | All 20 | All 20 (no re-test of an average) |

**Expected outcome:** circular far above nominal; the train/test split and FDR-BH near
nominal; LOO-CV with the analytic p-value still somewhat inflated (its assembled points are
dependent — fixed properly by the permutation test at the end).


In [ ]:
N_iter2 = 1000
alpha2 = 0.05

counts = {"circular": 0, "loo_cv": 0, "split": 0, "fdr_bh": 0}

for _ in range(N_iter2):
    DA_mc = sA * rng.standard_normal((Ngroups, Ncells))

    # Circular
    res_mc = stats.pearsonr(DA_mc, Perf[:, np.newaxis], axis=0)
    sel_mc = np.where((res_mc.statistic > 0.1) & (res_mc.pvalue < 0.05))[0]
    if sel_mc.size > 0:
        _, p_circ = stats.pearsonr(DA_mc[:, sel_mc].mean(axis=1), Perf)
        if p_circ < alpha2:
            counts["circular"] += 1

    # LOO-CV
    RCGA_cv_mc, _ = loo_cv_rcga(DA_mc, Perf)
    mask = ~np.isnan(RCGA_cv_mc)
    if mask.sum() >= 3:
        _, p_cv = stats.pearsonr(RCGA_cv_mc[mask], Perf[mask])
        if p_cv < alpha2:
            counts["loo_cv"] += 1

    # 50/50 train/test split
    tt = train_test_rcga(DA_mc, Perf)
    if tt is not None and tt[0].size >= 3:
        _, p_tt = stats.pearsonr(tt[0], tt[1])
        if p_tt < alpha2:
            counts["split"] += 1

    # FDR-BH: declare an effect if >=1 cell survives BH at q = alpha2
    if fdr_bh_cells(DA_mc, Perf, q=alpha2).size > 0:
        counts["fdr_bh"] += 1

for name, cnt in counts.items():
    print(f"  {name:10s}  FP rate = {cnt/N_iter2:.1%}  ({cnt}/{N_iter2})")
print(f"  nominal α  = {alpha2:.1%}")


In [ ]:
labels = ["Circular\n(double-dipping)", "LOO-CV\n(analytic p)",
          "Train/Test\nsplit", "FDR-BH\n(corrected)"]
rates2 = [counts[k] / N_iter2 for k in ("circular", "loo_cv", "split", "fdr_bh")]
colors2 = ["tab:red", "tab:green", "tab:blue", "tab:purple"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(labels, rates2, color=colors2, width=0.5, edgecolor="k")
ax.axhline(alpha2, color="k", linestyle="--", linewidth=1.2, label=f"Nominal α = {alpha2}")
ax.set_ylabel("False-positive rate (null data)")
ax.set_title(f"False Positive rate at α = {alpha2}   (N = {N_iter2} simulations)")
ax.set_ylim(0, max(rates2) * 1.3)
for bar, rate in zip(bars, rates2):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f"{rate:.1%}", ha="center", va="bottom", fontsize=11)
ax.legend()
plt.tight_layout()
plt.show()


# Question 2 — Standard analysis method, **with a real effect**

Now we repeat the entire simulation, but this time behavioral discrimination performance
really *does* modulate the activity of the **performance-correlated cells** (We simulate that **10% of the cells** are genuinely modulated by behavioral performance). For those cells we add, on top of the noise, a signal that is a scaled,
mean-zero function of each trial group's performance $P$:

$$P_\text{scaled} = \frac{k\, \sigma_A}{\operatorname{std}(P)}\,(P - \overline{P})$$

The constant `k` sets how big the behavioral effect is relative to noise:
`k=1` → the signal is comparable to the noise; `k=0.1` → performance only weakly affects it.
Here `k=0.2` gives a modest correlation. All other cells remain pure noise. Then the rest of
the analysis workflow is applied exactly as in Question 1.




In [ ]:

Nresponsive = round(0.1 * Ncells)
# Indices of the cells that are genuinely modulated by performance (only in the real-effect case).
CorrelatedCells = np.arange(0, Nresponsive)
UncorrelatedCells = np.arange(Nresponsive, Ncells)




In [ ]:
k = 0.2  # modest correlation between cell signal and performance
Pscaled = (k * sA / np.std(Perf, ddof=1)) * (Perf - np.mean(Perf))

DA_real = sA * rng.standard_normal((Ngroups, Ncells))
DA_real[:, CorrelatedCells] += Pscaled[:, np.newaxis]

r2 = np.empty(Ncells)
p2 = np.empty(Ncells)
for j in range(Ncells):
    r2[j], p2[j] = stats.pearsonr(DA_real[:, j], Perf)
SelectedCells_real = np.where((r2 > 0.1) & (p2 < 0.05))[0]
RCGA_real = DA_real[:, SelectedCells_real].mean(axis=1)
print(f"Number of 'selected' cells: {SelectedCells_real.size}")


## Plot the real-effect result

Now run the same standard (circular) analysis on the real-effect data and plot RCGA vs.
performance. This time there genuinely is an effect — but note the circular plot looks much
like the null one, which is exactly why we can't trust the circular method on its own.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
annotate_corr(ax, RCGA_real, Perf, "Standard Analysis Method, With Real Effect")
plt.tight_layout()
plt.show()


## Where are the "correlated" cells? (spatial view)

Because each cell has a random 2D position, we can plot the imaging field and highlight the
subset of cells that the circular workflow *selected* as correlated with performance. Here we
show the **real-effect** data: the truly performance-correlated cells (the first
`Nresponsive` cells) are outlined for reference, and the selected set should overlap them more
often than chance — though, since positions are random, the selection still looks spatially
unstructured.


In [ ]:
def plot_cells_2d(ax, selected, title, true_subset=None):
    """Plot all cells at their 2D positions, highlighting the `selected` subset."""
    ax.scatter(CellXY[:, 0], CellXY[:, 1], s=20, color="lightgray",
               label="all cells", zorder=1)
    if true_subset is not None:
        ax.scatter(CellXY[true_subset, 0], CellXY[true_subset, 1], s=90,
                   facecolors="none", edgecolors="tab:orange", linewidths=1.5,
                   label="truly correlated cells", zorder=2)
    ax.scatter(CellXY[selected, 0], CellXY[selected, 1], s=30, color="tab:red",
               label="selected cells", zorder=3)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.set_aspect("equal")
    ax.set_xlabel("x position")
    ax.set_ylabel("y position")
    ax.set_title(title)
    ax.legend(loc="upper right", fontsize=8)


fig, ax = plt.subplots(figsize=(7, 6))
plot_cells_2d(ax, SelectedCells_real,
              f"With Real Effect — {SelectedCells_real.size} selected cells",
              true_subset=CorrelatedCells)
plt.tight_layout()
plt.show()


## Do the valid methods still detect the real effect?

Now apply the same independent workflows — LOO-CV, the 50/50 train/test split, and FDR-BH — to
the **real-effect** data. Unlike the null case, here we *want* a detection. The valid methods
have lower power than the (invalid) circular method, but they should still pick up a genuine
effect, and their result can be trusted.


In [ ]:
# LOO-CV on the real-effect data
RCGA_cv_real, ncells_real = loo_cv_rcga(DA_real, Perf)
mask_real = ~np.isnan(RCGA_cv_real)
r_cv_real, p_cv_real = stats.pearsonr(RCGA_cv_real[mask_real], Perf[mask_real])

# 50/50 train/test split on the real-effect data
split_real = train_test_rcga(DA_real, Perf)

# FDR-BH on the real-effect data
fdr_real = fdr_bh_cells(DA_real, Perf, q=q_fdr)
true_hits = np.intersect1d(fdr_real, CorrelatedCells).size

print("Real-effect data (k = %.2f):" % k)
print(f"  LOO-CV        : r = {r_cv_real:+.3f}, p = {p_cv_real:.3f} "
      f"({mask_real.sum()}/{Ngroups} folds selected a cell)")
if split_real is None:
    print("  Train/test    : no cells selected on the training half")
else:
    r_tt_real, p_tt_real = stats.pearsonr(split_real[0], split_real[1])
    print(f"  Train/test    : r = {r_tt_real:+.3f}, p = {p_tt_real:.3f} "
          f"({split_real[2].size} cells selected on training half)")
print(f"  FDR-BH        : {fdr_real.size} cells discovered, "
      f"{true_hits}/{CorrelatedCells.size} are truly correlated")


### Rejecting the true positive?
This method prevents us from detecting the false positive in the no real effect dataset, but we also don't see a true positive result in the dataset with a real effect. What if we try increasing the strength of the real effect? 

In [ ]:
# simulate a stronger effect, k = 0.5
k_strong = 0.5  # stronger correlation between cell signal and performance
Pscaled_strong = (k_strong * sA / np.std(Perf, ddof=1)) * (Perf - np.mean(Perf))

DA_real_strong = sA * rng.standard_normal((Ngroups, Ncells))
DA_real_strong[:, CorrelatedCells] += Pscaled_strong[:, np.newaxis]

r2 = np.empty(Ncells)
p2 = np.empty(Ncells)
for j in range(Ncells):
    r2[j], p2[j] = stats.pearsonr(DA_real_strong[:, j], Perf)
SelectedCells_real_strong = np.where((r2 > 0.1) & (p2 < 0.05))[0]
RCGA_real_strong = DA_real_strong[:, SelectedCells_real_strong].mean(axis=1)
print(f"Number of 'selected' cells: {SelectedCells_real_strong.size}")


In [ ]:
RCGA_cv_real_strong, ncells_real_strong = loo_cv_rcga(DA_real_strong, Perf)

print("\nWeak real-effect data:")
print(f"  Avg cells selected per fold : {np.mean(ncells_real):.1f}  "
      f"(range {min(ncells_real)}–{max(ncells_real)})")
print(f"  Folds with ≥1 cell selected : {np.sum(np.array(ncells_real) > 0)}/{Ngroups}")

print("\nStrong real-effect data:")
print(f"  Avg cells selected per fold : {np.mean(ncells_real_strong):.1f}  "
      f"(range {min(ncells_real_strong)}–{max(ncells_real_strong)})")
print(f"  Folds with ≥1 cell selected : {np.sum(np.array(ncells_real_strong) > 0)}/{Ngroups}")


In [ ]:
# plot the results for LOO-CV - no effect, weak effect, strong effect
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

annotate_corr_cv(axes[0], RCGA_cv_null, Perf,
                 "LOO-CV — No Real Effect\n(independent selection & test)")
annotate_corr_cv(axes[1], RCGA_cv_real, Perf,
                 "LOO-CV — With Weak Real Effect\n(independent selection & test)")
annotate_corr_cv(axes[2], RCGA_cv_real_strong, Perf,
                 "LOO-CV — With Strong Real Effect\n(independent selection & test)")

plt.tight_layout()
plt.show()


---

## Power Curves: Effect Size and Sample Size Sweeps

Here we measure **power** — the true-positive rate on real-effect data — across two parameter sweeps:

1. **Effect-size sweep** — $k$ from 0.1 to 1.0 in steps of 0.1, with $N = 20$ trial groups fixed.
   Shows the minimum effect size each method can reliably detect.

2. **Sample-size sweep** — the number of trial groups $N$ from 5 to 25 in steps of 1, with
   $k = 0.2$ fixed.  Shows how many trial groups each method needs to reach adequate power.

For each combination, `N_iter_power` real-effect datasets are simulated and the
fraction of runs that produce a detection is recorded.  $P$ and the cell activity are
resampled fresh each run so the estimate reflects genuine variability.

> **Note on the circular method:** its "power" includes spurious detections arising
> from the circular selection bias — the high curve is not evidence of a trustworthy
> method, but of an inflated false-positive rate carrying over into real-effect data.



In [ ]:
def power_sweep(param_values, param_name, N_iter, alpha, k_fixed=0.2, N_fixed=20):
    """Estimate detection power for each value of k (effect size) or N (number of trial groups).

    Simulates real-effect data for each parameter value and returns the fraction
    of runs where each method correctly reaches a detection at level alpha.

    Parameters
    ----------
    param_values : array-like
    param_name   : 'k' or 'N'
    N_iter       : simulations per parameter value
    alpha        : significance threshold
    k_fixed      : effect size used when sweeping N
    N_fixed      : number of trial groups used when sweeping k
    """
    power = {m: [] for m in ("circular", "loo_cv", "fdr_bh")}

    for val in param_values:
        k_val = float(val) if param_name == "k" else k_fixed
        N_val = int(val)   if param_name == "N" else N_fixed
        counts = {"circular": 0, "loo_cv": 0, "fdr_bh": 0}

        for _ in range(N_iter):
            # Fresh performance scores for this run
            Perf_sim = np.round(sP * rng.standard_normal(N_val) + mP)
            std_P = np.std(Perf_sim, ddof=1)
            if std_P < 1e-10:
                continue

            # Real-effect cell data: performance-correlated cells modulated by performance
            Pscaled_sim = (k_val * sA / std_P) * (Perf_sim - np.mean(Perf_sim))
            DA_sim = sA * rng.standard_normal((N_val, Ncells))
            DA_sim[:, CorrelatedCells] += Pscaled_sim[:, np.newaxis]

            # Circular
            res_s = stats.pearsonr(DA_sim, Perf_sim[:, np.newaxis], axis=0)
            sel_s = np.where((res_s.statistic > 0.1) & (res_s.pvalue < 0.05))[0]
            if sel_s.size > 0:
                _, p_circ = stats.pearsonr(DA_sim[:, sel_s].mean(axis=1), Perf_sim)
                if p_circ < alpha:
                    counts["circular"] += 1

            # LOO-CV
            RCGA_cv_s, _ = loo_cv_rcga(DA_sim, Perf_sim)
            cv_mask = ~np.isnan(RCGA_cv_s)
            if cv_mask.sum() >= 3:
                _, p_cv = stats.pearsonr(RCGA_cv_s[cv_mask], Perf_sim[cv_mask])
                if p_cv < alpha:
                    counts["loo_cv"] += 1

            # FDR-BH: detection if >=1 cell survives BH at q = alpha
            if fdr_bh_cells(DA_sim, Perf_sim, q=alpha).size > 0:
                counts["fdr_bh"] += 1

        for m in power:
            power[m].append(counts[m] / N_iter)

    return power


In [ ]:
N_iter_power = 200   # increase to 500–1000 for smoother curves (takes longer)
alpha_power  = 0.05  # standard threshold for power analysis

k_values = np.round(np.arange(0.1, 1.01, 0.1), 2)   # [0.1, 0.2, ..., 1.0]
N_values = np.arange(5, 26, 1)                        # [5, 6, ..., 25] trial groups

print(f"Running effect-size sweep ({len(k_values)} k values × {N_iter_power} iterations) …")
power_k = power_sweep(k_values, "k", N_iter_power, alpha_power, N_fixed=20)
print("  done.")

print(f"Running sample-size sweep ({len(N_values)} N values × {N_iter_power} iterations) …")
power_N = power_sweep(N_values, "N", N_iter_power, alpha_power, k_fixed=0.2)
print("  done.")


In [ ]:
method_styles = {
    "circular": ("tab:red",    "Circular (double-dipping)"),
    "loo_cv":   ("tab:green",  "LOO-CV"),
    "fdr_bh":   ("tab:purple", "FDR-BH (corrected)"),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: power vs effect size k (N=20 trial groups fixed)
ax = axes[0]
for m, (color, label) in method_styles.items():
    ax.plot(k_values, power_k[m], "o-", color=color, label=label, linewidth=2)
ax.axhline(alpha_power, color="k", linestyle="--", linewidth=1.2,
           label=f"α = {alpha_power} (nominal FP rate)")
ax.set_xlabel("Effect size  k")
ax.set_ylabel("Detection rate (power)")
ax.set_title(f"Power vs Effect Size\n(N = 20 trial groups,  {N_iter_power} simulations per k,  α = {alpha_power})")
ax.set_xlim(k_values[0] - 0.05, k_values[-1] + 0.05)
ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: power vs number of trial groups N (k=0.2 fixed)
ax = axes[1]
for m, (color, label) in method_styles.items():
    ax.plot(N_values, power_N[m], "o-", color=color, label=label, linewidth=2)
ax.axhline(alpha_power, color="k", linestyle="--", linewidth=1.2,
           label=f"α = {alpha_power} (nominal FP rate)")
ax.set_xlabel("Number of trial groups  N")
ax.set_ylabel("Detection rate (power)")
ax.set_title(f"Power vs Number of Trial Groups\n(k = 0.2,  {N_iter_power} simulations per N,  α = {alpha_power})")
ax.set_xlim(N_values[0] - 0.5, N_values[-1] + 0.5)
ax.set_ylim(-0.02, 1.05)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---

## A better use of cross-validation: permutation-based testing

The LOO-CV false-positive rate above is still well above nominal. The problem is not the
cross-validation itself but the **final test**: feeding the assembled `RCGA_cv` vector into
`stats.pearsonr` assumes 20 *independent* observations, whereas the folds share almost all
their data and reuse the same fixed `Perf`, so the values are dependent and the analytic
p-value is anti-conservative.

The fix: keep the cross-validated statistic, but get its significance from a
**label-permutation null** instead of the t-distribution. Shuffling `Perf` and re-running the
*entire* selection + CV pipeline reproduces the same dependence and selection artifact under
the null, so comparing the observed statistic to that null is calibrated.

*(These cells use a separate `perm_rng` so they do not alter the global random stream or any
results above, and take a couple of minutes to run.)*


In [ ]:
# A cross-validated statistic is (approximately) unbiased, but its significance must come from
# a permutation null, not the analytic pearsonr p-value (which assumes independent points).
perm_rng = np.random.default_rng(1)  # local generator: keeps the global rng / results above unchanged


def loo_cv_corr(DA, Perf):
    """The scalar LOO-CV statistic: correlation of held-out RCGA with performance."""
    rcga, _ = loo_cv_rcga(DA, Perf)
    mask = ~np.isnan(rcga)
    if mask.sum() < 3:
        return 0.0
    return stats.pearsonr(rcga[mask], Perf[mask]).statistic


def loo_cv_perm_pvalue(DA, Perf, n_perm=1000, generator=perm_rng):
    """Significance of the LOO-CV correlation via label permutation.

    Shuffling `Perf` re-runs the *entire* selection + CV pipeline, so the null carries the
    same fold dependence and selection artifact that inflate the analytic p-value. One-sided
    because selection keeps positive-correlation cells (r > 0.1).
    """
    observed = loo_cv_corr(DA, Perf)
    null = np.array([loo_cv_corr(DA, generator.permutation(Perf)) for _ in range(n_perm)])
    p = (1 + np.sum(null >= observed)) / (1 + n_perm)
    return observed, p


for name, D in [("null (no effect)", DA),
                (f"weak effect (k={k})", DA_real),
                (f"strong effect (k={k_strong})", DA_real_strong)]:
    obs, pval = loo_cv_perm_pvalue(D, Perf, n_perm=1000)
    print(f"{name:22s}  observed CV r = {obs:+.3f}   permutation p = {pval:.3f}")


### False-positive rate: analytic vs permutation p-value

Now repeat the null Monte-Carlo (no real effect) and compare, for each dataset, the LOO-CV
decision made with the **analytic** `pearsonr` p-value versus the **permutation** p-value.

**Expected outcome:** the analytic test stays inflated (dependent CV points break its
independence assumption), while the permutation test returns to the nominal α.


In [ ]:
# To make ~40,000 cross-validated fits tractable we use a fast but *mathematically identical*
# LOO: with 19 training groups the rule "r > 0.1 AND p < 0.05" is exactly "r > r_crit",
# because the two-sided p < 0.05 threshold (|r| > 0.456) already implies r > 0.1.
_tc = stats.t.ppf(1 - 0.05 / 2, (Ngroups - 1) - 2)
R_CRIT = _tc / np.sqrt((Ngroups - 1) - 2 + _tc ** 2)


def _corr(a, b):
    a = a - a.mean()
    b = b - b.mean()
    return (a * b).sum() / np.sqrt((a ** 2).sum() * (b ** 2).sum())


def loo_cv_rcga_fast(DA, Perf):
    """Same LOO-CV RCGA as loo_cv_rcga, but correlation-only (no p-value) for speed."""
    Ng = DA.shape[0]
    RCGA = np.full(Ng, np.nan)
    for i in range(Ng):
        tr = np.delete(np.arange(Ng), i)
        y = Perf[tr] - Perf[tr].mean()
        Xc = DA[tr] - DA[tr].mean(axis=0)
        r = (Xc * y[:, None]).sum(0) / np.sqrt((Xc ** 2).sum(0) * (y ** 2).sum())
        sel = np.where(r > R_CRIT)[0]
        if sel.size > 0:
            RCGA[i] = DA[i, sel].mean()
    return RCGA


def loo_cv_corr_fast(DA, Perf):
    rcga = loo_cv_rcga_fast(DA, Perf)
    mask = ~np.isnan(rcga)
    if mask.sum() < 3:
        return 0.0
    return _corr(rcga[mask], Perf[mask])


N_datasets = 200   # null datasets (no real effect)
n_perm_fp  = 200   # label permutations per dataset
alpha_perm = 0.05  # nominal false-positive rate

fp_analytic = 0
fp_perm = 0
for _ in range(N_datasets):
    DA_null = sA * perm_rng.standard_normal((Ngroups, Ncells))
    rcga = loo_cv_rcga_fast(DA_null, Perf)
    mask = ~np.isnan(rcga)
    if mask.sum() < 3:
        continue
    obs = _corr(rcga[mask], Perf[mask])

    # (1) analytic, one-sided (positive) p from pearsonr on the assembled CV vector
    _, p_two = stats.pearsonr(rcga[mask], Perf[mask])
    p_analytic = p_two / 2 if obs > 0 else 1.0
    if p_analytic < alpha_perm:
        fp_analytic += 1

    # (2) permutation p: shuffle Perf, rerun the whole selection + CV pipeline
    null = np.array([loo_cv_corr_fast(DA_null, perm_rng.permutation(Perf))
                     for _ in range(n_perm_fp)])
    if (1 + np.sum(null >= obs)) / (1 + n_perm_fp) < alpha_perm:
        fp_perm += 1

print(f"LOO-CV false-positive rate on null data "
      f"({N_datasets} datasets, α = {alpha_perm:.0%}):")
print(f"  analytic pearsonr p  : {fp_analytic / N_datasets:.1%}")
print(f"  permutation p        : {fp_perm / N_datasets:.1%}")
print(f"  nominal α            : {alpha_perm:.0%}")
